# 3. Decision Tree

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [2]:
# --- Load and preprocess churn dataset (Classification) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_churn = pd.read_csv('churn_df.csv')
# Dropping ID columns which have no predictive value
df_churn = df_churn.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')

X_clf = df_churn.drop(columns=['Exited'])
y_clf = df_churn['Exited']

# Handling missing values
X_clf.fillna(X_clf.mean(numeric_only=True), inplace=True)
for col in X_clf.select_dtypes(include=['object']).columns:
    X_clf[col].fillna(X_clf[col].mode()[0], inplace=True)

# Categorical mapping using get_dummies
X_clf = pd.get_dummies(X_clf, drop_first=True)

# Train Test Split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Scaling
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)
print("Churn dataset ready for classification!")


Churn dataset ready for classification!


In [3]:
# --- Load and preprocess housing dataset (Regression) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_housing = pd.read_csv('housing.csv')
X_reg = df_housing.drop(columns=['median_house_value'])
y_reg = df_housing['median_house_value']

# Handling missing values
X_reg.fillna(X_reg.mean(numeric_only=True), inplace=True)
for col in X_reg.select_dtypes(include=['object']).columns:
    X_reg[col].fillna(X_reg[col].mode()[0], inplace=True)

# Categorical mapping using get_dummies
X_reg = pd.get_dummies(X_reg, drop_first=True)

# Train Test Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Scaling
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)
print("Housing dataset ready for regression!")


Housing dataset ready for regression!


## ID3 Classification (using entropy) - Churn Data
Optimization: Added `class_weight='balanced'` to massively improve Class 1 recall.

In [4]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

id3_clf = DecisionTreeClassifier(criterion='entropy', max_depth=6, class_weight='balanced', random_state=42)
id3_clf.fit(X_train_clf_scaled, y_train_clf)

print("ID3 Classification Accuracy:", accuracy_score(y_test_clf, id3_clf.predict(X_test_clf_scaled)))
print(classification_report(y_test_clf, id3_clf.predict(X_test_clf_scaled)))


ID3 Classification Accuracy: 0.7455
              precision    recall  f1-score   support

           0       0.94      0.73      0.82      1607
           1       0.42      0.81      0.56       393

    accuracy                           0.75      2000
   macro avg       0.68      0.77      0.69      2000
weighted avg       0.84      0.75      0.77      2000



## CART Classification (using gini) - Churn Data
Optimization: Added `class_weight='balanced'`

In [5]:
cart_clf = DecisionTreeClassifier(criterion='gini', max_depth=6, class_weight='balanced', random_state=42)
cart_clf.fit(X_train_clf_scaled, y_train_clf)

print("CART Classification Accuracy:", accuracy_score(y_test_clf, cart_clf.predict(X_test_clf_scaled)))
print(classification_report(y_test_clf, cart_clf.predict(X_test_clf_scaled)))


CART Classification Accuracy: 0.7455
              precision    recall  f1-score   support

           0       0.93      0.73      0.82      1607
           1       0.42      0.79      0.55       393

    accuracy                           0.75      2000
   macro avg       0.68      0.76      0.69      2000
weighted avg       0.83      0.75      0.77      2000



## CART Regression (using squared_error) - Housing Data
Optimization: Deepened `max_depth` to 10 and calculated RMSE instead of MSE.

In [7]:
cart_reg = DecisionTreeRegressor(criterion='squared_error', max_depth=10, random_state=42)
cart_reg.fit(X_train_reg_scaled, y_train_reg)

rmse = np.sqrt(mean_squared_error(y_test_reg, cart_reg.predict(X_test_reg_scaled)))
print(f"CART Regression RMSE: {rmse:,.2f}")
print("CART Regression R2:", r2_score(y_test_reg, cart_reg.predict(X_test_reg_scaled)))


CART Regression RMSE: 61,441.76
CART Regression R2: 0.7119151283417915
